In [2]:
# ------------------------------------------------------------------
# Import Required Libraries
#
# pathlib : Handles file system paths in a platform-independent way.
# pandas  : Loads and analyzes tabular datasets.
# numpy   : Supports numerical operations used throughout the project.
# ------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------------
# Configure Project Directories
#
# Define the root project directory and the location of the source
# datasets used throughout the notebook.
# ------------------------------------------------------------------

PROJECT_PATH = Path("..")

DATASET_PATH = PROJECT_PATH / "datasets"

ARTIFACT_PATH = PROJECT_PATH / "artifacts"


# ------------------------------------------------------------------
# Discover Available Datasets
#
# Search recursively for all CSV files available in the datasets
# directory and display the discovered source files.
# ------------------------------------------------------------------

csv_files = sorted(DATASET_PATH.rglob("*.csv"))

print(f"Datasets discovered: {len(csv_files)}\n")

for file in csv_files:
    print(file.relative_to(DATASET_PATH))

Datasets discovered: 9

commerce\raw\olist_order_items_dataset.csv
commerce\raw\olist_order_payments_dataset.csv
commerce\raw\olist_orders_dataset.csv
commerce\raw\olist_products_dataset.csv
commerce\raw\olist_sellers_dataset.csv
commerce\raw\product_category_name_translation.csv
custommer\raw\olist_customers_dataset.csv
custommer\raw\olist_order_reviews_dataset.csv
delivery\raw\olist_geolocation_dataset.csv


In [3]:
# ------------------------------------------------------------------
# Define Semantic Column Mapping
#
# Map equivalent business attributes that use different physical
# column names across datasets.
# ------------------------------------------------------------------

semantic_mapping = {

    "customer_id": [
        "customer_id"
    ],

    "order_id": [
        "order_id"
    ],

    "product_id": [
        "product_id"
    ],

    "seller_id": [
        "seller_id"
    ],

    "postal_code": [
        "customer_zip_code_prefix",
        "seller_zip_code_prefix",
        "geolocation_zip_code_prefix"
    ]

}

In [4]:
# ------------------------------------------------------------------
# Get Business Attribute
#
# Return the business attribute associated with a physical column.
# ------------------------------------------------------------------

def get_business_attribute(column_name):

    for attribute, columns in semantic_mapping.items():

        if column_name in columns:
            return attribute

    return None

In [5]:
# ------------------------------------------------------------------
# Load Datasets
#
# Load all discovered datasets into memory.
# ------------------------------------------------------------------

datasets = {}

for file in csv_files:

    datasets[file.stem] = pd.read_csv(file)

In [6]:
# ------------------------------------------------------------------
# Discover Dataset Relationships
#
# Identify potential relationships between datasets by analyzing
# common column names and value overlap.
#
# This discovery process will support future PK/FK definitions,
# dimensional modeling, and Lakehouse architecture decisions.
# ------------------------------------------------------------------

relationships = []

for source_name, source_df in datasets.items():

    for target_name, target_df in datasets.items():

        if source_name == target_name:
            continue

        common_columns = set(source_df.columns).intersection(
            set(target_df.columns)
        )

        for column in common_columns:

            source_values = set(
                source_df[column]
                .dropna()
                .astype(str)
                .unique()
            )

            target_values = set(
                target_df[column]
                .dropna()
                .astype(str)
                .unique()
            )

            if len(source_values) == 0:
                continue

            overlap = (
                len(source_values.intersection(target_values))
                / len(source_values)
            )

            if overlap > 0.5:

                relationships.append({

                    "Relationship Type": "Structural",

                    "Source Dataset": source_name,

                    "Source Column": column,

                    "Target Dataset": target_name,

                    "Target Column": column,

                    "Business Attribute": get_business_attribute(column),

                    "Match Percentage (%)": round(
                        overlap * 100,
                        2
                    )

                })

In [7]:
# ------------------------------------------------------------------
# Discover Semantic Relationships
#
# Identify relationships based on equivalent business attributes.
# ------------------------------------------------------------------

for source_name, source_df in datasets.items():

    for target_name, target_df in datasets.items():

        if source_name == target_name:
            continue

        for source_column in source_df.columns:

            source_attribute = get_business_attribute(source_column)

            if source_attribute is None:
                continue

            for target_column in target_df.columns:

                target_attribute = get_business_attribute(target_column)

                if source_attribute != target_attribute:
                    continue

                if source_column == target_column:
                    continue

                source_values = set(
                    source_df[source_column]
                    .dropna()
                    .astype(str)
                    .unique()
                )

                target_values = set(
                    target_df[target_column]
                    .dropna()
                    .astype(str)
                    .unique()
                )

                if len(source_values) == 0:
                    continue

                overlap = (
                    len(source_values.intersection(target_values))
                    / len(source_values)
                )

                if overlap > 0.5:

                    relationships.append({

                        "Relationship Type": "Semantic",

                        "Source Dataset": source_name,

                        "Source Column": source_column,

                        "Target Dataset": target_name,

                        "Target Column": target_column,

                        "Business Attribute": source_attribute,

                        "Match Percentage (%)": round(
                            overlap * 100,
                            2
                        )

                    })

In [8]:
# ------------------------------------------------------------------
# Build Enterprise Relationship Discovery
#
# Create the final relationship catalog.
# ------------------------------------------------------------------

enterprise_relationship_discovery = (
    pd.DataFrame(relationships)
    .sort_values(
        by=[
            "Relationship Type",
            "Source Dataset",
            "Target Dataset",
            "Source Column"
        ]
    )
    .reset_index(drop=True)
)

enterprise_relationship_discovery

,Relationship Type,Source Dataset,Source Column,Target Dataset,Target Column,Business Attribute,Match Percentage (%)
0,Semantic,olist_customers_dataset,customer_zip_code_prefix,olist_geolocation_dataset,geolocation_zip_code_prefix,postal_code,98.95
1,Semantic,olist_geolocation_dataset,geolocation_zip_code_prefix,olist_customers_dataset,customer_zip_code_prefix,postal_code,78.03
2,Semantic,olist_sellers_dataset,seller_zip_code_prefix,olist_customers_dataset,customer_zip_code_prefix,postal_code,96.26
3,Semantic,olist_sellers_dataset,seller_zip_code_prefix,olist_geolocation_dataset,geolocation_zip_code_prefix,postal_code,99.69
4,Structural,olist_customers_dataset,customer_id,olist_orders_dataset,customer_id,customer_id,100.00
5,Structural,olist_order_items_dataset,order_id,olist_order_payments_dataset,order_id,order_id,100.00
6,Structural,olist_order_items_dataset,order_id,olist_order_reviews_dataset,order_id,order_id,99.24
7,Structural,olist_order_items_dataset,order_id,olist_orders_dataset,order_id,order_id,100.00
8,Structural,olist_order_items_dataset,product_id,olist_products_dataset,product_id,product_id,100.00
9,Structural,olist_order_items_dataset,seller_id,olist_sellers_dataset,seller_id,seller_id,100.00


In [9]:
# ------------------------------------------------------------------
# Export Enterprise Relationship Discovery
#
# Persist the relationship catalog as a reusable data asset.
#
# This artifact documents discovered relationships between datasets,
# including structural and semantic relationships.
#
# It will support future data modeling, Lakehouse design,
# and enterprise architecture decisions.
# ------------------------------------------------------------------

relationship_path = ARTIFACT_PATH / "relationships"

relationship_path.mkdir(
    parents=True,
    exist_ok=True
)

enterprise_relationship_discovery.to_csv(
    relationship_path / "enterprise_relationship_discovery.csv",
    index=False
)